# Women's Safety Alert System — ML Prototype

This beginner project compares Decision Tree and K-Nearest Neighbors (KNN) using synthetic example scenarios. **Safety limitation:** this dataset is artificial and its labels are generated by a teaching rule. It does not measure real danger. This notebook does not send SOS alerts or contact police. A model must never delay a user's manual SOS request.

## 1. Idea
A future women's safety app could let a user press SOS and share location with trusted contacts or an authorized emergency service. This notebook covers only the ML assignment: learning from synthetic examples and comparing classifiers.

## 2. Load and inspect dataset
Keep `synthetic_safety_scenarios.csv` beside this notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = pd.read_csv('synthetic_safety_scenarios.csv')
display(df.head())
print('Shape:', df.shape)
print('Missing values:\n', df.isnull().sum())
print('Target counts (0=routine, 1=elevated concern):\n', df['elevated_concern_label'].value_counts())

### Prepare features and target
Input columns describe synthetic scenario conditions. Target `elevated_concern_label`: 0 = routine, 1 = elevated concern. These are not validated real-world safety indicators.

In [ ]:
X = df.drop(columns=['elevated_concern_label'])
y = df['elevated_concern_label']
assert X.isnull().sum().sum() == 0
assert y.isnull().sum() == 0
print('Features:', X.shape, 'Target:', y.shape)

## 3. Split data and train two models
Use an 80/20 stratified split. KNN uses standard scaling because it relies on distances.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
models = {
 'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
 'KNN': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
}
print('Training rows:', len(X_train), '| Testing rows:', len(X_test))

## 4. Evaluate accuracy and confusion matrices

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
scores = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    scores[name] = accuracy_score(y_test, pred)
    print(f'\n{name} accuracy: {scores[name]:.3f}')
    print(classification_report(y_test, pred, labels=[0,1], target_names=['routine','elevated concern'], zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=[0,1], display_labels=['routine','elevated concern'])
    plt.title(f'{name} — Confusion Matrix'); plt.tight_layout(); plt.show()
results = pd.DataFrame({'Model': list(scores.keys()), 'Accuracy': list(scores.values())})
display(results.sort_values('Accuracy', ascending=False))
results.plot(kind='bar', x='Model', y='Accuracy', legend=False, ylim=(0,1), rot=0)
plt.ylabel('Accuracy'); plt.title('Accuracy comparison (synthetic data)'); plt.tight_layout(); plt.show()
best = max(scores, key=scores.get)
print(f'Highest accuracy on this test split: {best} ({scores[best]:.1%})')

## 5. Conclusion
Use the printed results to state which model scored higher on this test split. Accuracy and confusion matrices compare model predictions, but they do not establish real-world safety performance because this dataset is synthetic.

**What I learned:** loading data, checking missing values, preparing features and target, splitting data, training two classifiers, and evaluating accuracy and confusion matrices.

### Safety and ethics
- This project does not detect actual danger or contact police.
- Never let a prediction delay a manual SOS request.
- A real SOS app needs user permission for location, secure data handling, reliable alert delivery, and authorized emergency-service integration.
- Test only with a simulation or consenting contacts; do not send fake emergency alerts.